In [1]:
import os
import sys
from pathlib import Path

sys.path.append("../")

In [2]:
from src.common import tools

config = tools.load_config()

In [3]:
from src.data import download

data_path = Path("../data")

if os.path.exists(data_path / config["magnus_carlsen_games_dataset_name"]):
    print(
        f"Dataset {config['magnus_carlsen_games_dataset_name']} already exists at {data_path}. Skipping download."
    )
else:
    download.kaggle_download_data(
        config["magnus_carlsen_games_dataset_handle"],
        data_path,
        config["magnus_carlsen_games_dataset_name"],
    )

Dataset magnus_carlsen_games already exists at ../data. Skipping download.


In [4]:
import pandas as pd

games_df = (
    pd.read_csv(
        "../data/magnus_carlsen_games/magnus_carlsen_all_online_games_cleaned.csv",
        usecols=["moves", "result_raw", "player_color"],
    )
    .dropna()
    .sample(n=5000, random_state=42)
    .reset_index(drop=True)
)

games_df["result_raw"] = games_df["result_raw"].map({"1-0": 1, "0-1": -1, "0.5-0.5": 0})
games_df["player_color"] = games_df.apply(
    lambda row: row["player_color"].lower(), axis=1
)

In [5]:
games_df.head()

,player_color,result_raw,moves
0,white,1,d4 Nf6 Nf3 c5 d5 g6 Nc3 Bg7 e4 d6 Bb5+ Bd7 a4 ...
1,white,1,1. e4 c5 2. Nf3 e6 3. c3 Nc6 4. d4 cxd4 5. cxd...
2,black,1,1. d4 d5 2. c4 dxc4 3. e4 Nf6 4. e5 Nd5 5. Bxc...
3,black,-1,e4 g6 d3 Bg7 Be3 Bxb2 Nc3 Bxc3+ Ke2 Bxa1 c3 f6...
4,white,-1,d4 Nf6 c4 g6 Nc3 Bg7 e4 d6 Be2 O-O Nf3 e5 O-O ...


In [6]:
games_df["result_raw"] = games_df.apply(
    lambda row: (
        row["result_raw"] if row["player_color"] == "white" else -row["result_raw"]
    ),
    axis=1,
)

In [7]:
games_df.head()

,player_color,result_raw,moves
0,white,1,d4 Nf6 Nf3 c5 d5 g6 Nc3 Bg7 e4 d6 Bb5+ Bd7 a4 ...
1,white,1,1. e4 c5 2. Nf3 e6 3. c3 Nc6 4. d4 cxd4 5. cxd...
2,black,-1,1. d4 d5 2. c4 dxc4 3. e4 Nf6 4. e5 Nd5 5. Bxc...
3,black,1,e4 g6 d3 Bg7 Be3 Bxb2 Nc3 Bxc3+ Ke2 Bxa1 c3 f6...
4,white,-1,d4 Nf6 c4 g6 Nc3 Bg7 e4 d6 Be2 O-O Nf3 e5 O-O ...


In [8]:
expanded_df = tools.expand_game_positions_san(
    games_df, moves_col="moves", eval_col="result_raw"
)

 31%|███       | 1554/5000 [00:06<00:14, 240.90it/s]


KeyboardInterrupt: 

In [10]:
expanded_df.head()

,fen,move,evaluation
0,rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w ...,d2d4,1
1,rnbqkbnr/pppppppp/8/8/3P4/8/PPP1PPPP/RNBQKBNR ...,g8f6,-1
2,rnbqkb1r/pppppppp/5n2/8/3P4/8/PPP1PPPP/RNBQKBN...,g1f3,1
3,rnbqkb1r/pppppppp/5n2/8/3P4/5N2/PPP1PPPP/RNBQK...,c7c5,-1
4,rnbqkb1r/pp1ppppp/5n2/2p5/3P4/5N2/PPP1PPPP/RNB...,d4d5,1


In [6]:
expanded_df["evaluation"] = None

In [7]:
from src.preprocess import datasets

dataset = datasets.ChessDataset(
    expanded_df,
)

In [8]:
dataset[0][0].shape, dataset[0][1].shape, dataset[0][2].shape

(torch.Size([18, 8, 8]), torch.Size([20480]), torch.Size([1]))